# GNS Climate Project - LLM Corpus Analysis
Hadleigh Tiddy / Summer 2023\
\
Currently under construction.

## Loading the environment

First, we need to load dependencies. Packages should already be installed but if not, uncomment the first line. 

In [61]:
# Standard Library Imports
import time
from os import truncate

# Third-party Library Imports
from dotenv import load_dotenv
from IPython.display import display, HTML
import pandas as pd
import random
import tiktoken

# Local Imports
from retriever import (
    retrieve_txt_files,
    count_txt_files,
    retrieve_txt_files_with_length,
    retrieve_specific_file
)

#Langchain Imports
from langchain.llms import OpenAI
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain
from langchain import FewShotPromptTemplate
from langchain.agents.agent_types import AgentType
from langchain.chat_models import ChatOpenAI
from langchain_experimental.agents.agent_toolkits import create_pandas_dataframe_agent

#A function for counting the tokens in a text, to not exceed max number of tokens. 
def get_token_count(text):
    encoding = tiktoken.get_encoding("cl100k_base")
    return len(encoding.encode(text))

Load the the API key as an environment variable. Remember this can be changed in the .env file.

In [27]:
load_dotenv()

True

Set the corpus directory and the files we want to analyze. 

In [86]:
#To add more corpora, put them here

original= 'Original Corpus'
gabrielle = 'Gabrielle Corpus'
gita = 'Gita Corpus'

In [104]:
#Set the corpus you'd like to work with here
corpus = original

text_files_contents = retrieve_txt_files(corpus)
print("The corpus we're working with is called", corpus)
count_txt_files(corpus)

The corpus we're working with is called Original Corpus
There are 166 txt files in the directory.


Now we need to look at the length of the files (in terms of words and tokens). This is because there is a limit to how big the request can be.

In [105]:
file_lengths = retrieve_txt_files_with_length(corpus)
columns = ["title", "words", "tokens"]
df = pd.DataFrame(file_lengths, columns=columns)
df

,Title,Length (words)
0,NZHerald_27_06_2015_NZ_storm.txt,2871
1,Newsroom_West_Coast_article1.txt,2561
2,NZHerald_17_08_2022_Nelson.txt,2424
3,Stuff_21_06_2013_L.N.Island(2).txt,2104
4,Stuff_12_04_2017_Cook.txt,2019
...,...,...
161,Stuff_11_03_2014_Canterbury.txt,184
162,One_News_26_02_2023_Gabrielle.txt,169
163,Scoop_10_09_2013_U.N.Island.txt,151
164,NZHerald_23_07_2020_N.Island.txt,78


### Building a template

The following cells build a template prompt to be processed by the LLM.
If you want to change the output it's giving, do that here.

NOTE: If you change the template (for instance, to include a new field to collect), you'll need to update the cell which converts the output into a dataframe.

In [106]:
template = """
You're a researcher analysing articles to see how climate change is framed. 
You are assessing each article for key information and writing an analysis in a specific format. 
Only give the analysis, not the article, following the exact format provided below:

------------------------

ANALYSIS:

Date: 7/21/2021
News Outlet: Stuff
Region Reported On: West Coast
Name of Weather Event: Cyclone Gita
Weather Type: Cyclone
Mention of 'climate change': No
Number of times 'climate change' is mentioned: 0
Significant people (and roles) mentioned: Associate Housing Minister Poto Williams
Scientists mentioned: N/A
One sentence from text that includes 'climate change': N/A
Key theme (max 10 words): Temporary village in Westport for flood-displaced residents, various accommodations.

----------------------------------

Now do an analysis for the following article. Follow the same analysis format from above exactly. If you can't find an answer, use N/A:

{article}

-----------------------------------
"""


The next cell combines the prompt and the LLM we want to use in a 'chain'.

In [107]:
prompt = PromptTemplate.from_template(template)
llm=OpenAI(temperature=0.01, model="gpt-3.5-turbo-instruct")
OpenAI_chain = LLMChain(llm=llm, prompt=prompt)

Finally, we define a function that checks the token limits aren't exceeded before making a request.

In [108]:
def run_langchain(text):
    #The total tokens for question + response based on the model. This can change with different models!
    max_tokens = 4096
    
    #We have to anticipate how long answers will be - my expected response is about 250 tokens so I'm being overly careful here:
    expected_response_tokens = 400
    
    token_count = get_token_count(text)
    template_count = get_token_count(template)
    
    if (token_count + template_count) <= (max_tokens - expected_response_tokens):
        return OpenAI_chain.run({"article": text})
    else:
        return "Text exceeds the token limit ({} tokens, {} is maximum)".format((token_count + template_count + expected_response_tokens), max_tokens)

### Testing the template

The following cell picks a random text from the corpus to test if we are getting the typep of response we want. Have a look at the output to see if it is in the format required.

In [109]:
random_text = random.choice(text_files_contents)

result = run_langchain(random_text)
print("Example of a random text from the {}:".format(corpus))
print(result)

Example of a random text from the Original Corpus corpus

ANALYSIS:

Date: 3/3/2012
News Outlet: Stuff
Region Reported On: North Island
Name of Weather Event: Weather Bomb
Weather Type: Storm
Mention of 'climate change': No
Number of times 'climate change' is mentioned: 0
Significant people (and roles) mentioned: N/A
Scientists mentioned: N/A
One sentence from text that includes 'climate change': N/A
Key theme (max 10 words): Severe storm causes widespread damage and power outages.


The next cell allows us to analyze any particular file we want. Copy and paste in the name of the file with the largest token count to see if it works - if not, you might need to remove some of the larger files before proceeding. 

In [110]:
specific_file_name = 'NZHerald_27_06_2015_NZ_storm.txt'
specific_text = retrieve_specific_file(corpus, specific_file_name)

if specific_text is not None:
    result = run_langchain(specific_text)
    print(result)
else:
    print("File not found.")

Text exceeds the token limit (4388 tokens, 4096 is maximum)


Now we run that request for all available files, and instead of printing strings, we split the strings and put each answer into a pandas dataframe. This will do potentially a lot of requests of the LLM and take a long time - use sparingly!

With the 2 seconds of sleeptime, Turbo can do about 16 per minute: 10 mins for the original corpus. 

NOTE: I have saved one run of 160 articles to a pickle - see cells below.

In [111]:
##DON'T RUN THIS UNLESS YOU'RE SURE WHAT YOU'RE DOING!
#This cell will process all of the texts in the corpus, appending the results to a dataframe. 

columns = []
data = []
count = 0
count_txt_files(corpus)

for text in text_files_contents:

    # Get the analysis
    analysis_result = run_langchain(text)

    if not analysis_result.startswith("Text exceeds the token limit"):

        # Split up the text based on colons and append to the dictionary
        result_dict = {col: 'N/A' for col in columns}  # Initialize with 'N/A'
        for line in analysis_result.split('\n'):
            if ': ' in line:
                key, value = line.split(': ', 1)
                key = key.strip()
                value = value.strip()

                # If the key is not already in the columns list, add it
                if key not in columns:
                    columns.append(key)

                # Update the result dictionary
                result_dict[key] = value

        # Add the parsed data to the list
        data.append(result_dict)
        count += 1
        print('Processed', count)
        corpus_dataframe = pd.DataFrame(data, columns=columns)
        corpus_dataframe[]
    
    else:
        lines = text.splitlines()
        title = lines[0]
        print('Excluded {} - too long'.format(title))
        count += 1



There are 166 txt files in the directory.
Processed 1
Processed 2
Processed 3
Processed 4
Processed 5
Processed 6
Processed 7
Processed 8
Processed 9
Processed 10
Processed 11
Processed 12
Processed 13
Processed 14
Processed 15
Processed 16
Processed 17
Processed 18
Processed 19
Processed 20
Processed 21
Processed 22
Processed 23
Processed 24
Processed 25
Processed 26
Processed 27
Processed 28
Processed 29
Processed 30
Processed 31
Processed 32
Processed 33
Processed 34
Processed 35
Processed 36
Processed 37
Processed 38
Processed 39
Processed 40
Processed 41
Processed 42
Processed 43
Processed 44
Processed 45
Processed 46
Processed 47
Processed 48
Processed 49
Processed 50
Processed 51
Processed 52
Processed 53
Processed 54
Processed 55
Processed 56
Processed 57
Processed 58
Processed 59
Processed 60
Processed 61
Processed 62
Processed 63
Processed 64
Processed 65
Processed 66
Processed 67
Processed 68
Processed 69
Processed 70
Processed 71
Processed 72
Processed 73
Processed 74
Proce

In [116]:
corpus_dataframe

,Date,News Outlet,Region Reported On,Name of Weather Event,Weather Type,Mention of 'climate change',Number of times 'climate change' is mentioned,Significant people (and roles) mentioned,Scientists mentioned,One sentence from text that includes 'climate change',Key theme (max 10 words)
0,3/2/2021,Stuff,Southland,Flooding,Heavy Rain,Yes,4,Emergency Management Southland team leader Cra...,NIWA climate change scientists,"""If there is one stark lesson from the floodin...",Impact of climate change on flooding in Southl...
1,7/21/2021,RNZ,West Coast,Westport Flood,Flood,No,0,"Westport man Pat Koti, Buller Mayor Jamie Clei...",N/A,N/A,Westport flood victims struggle with housing a...
2,1/28/2023,Newstalkzb,Auckland,Auckland Floods,Heavy Rainfall,Yes,6,Victoria University climate scientist Professo...,Lisa Murray,"""But those figures are almost certainly undere...",Extreme rainfall event and its connection to c...
3,11/20/2018,Stuff,South Island,Ex-Cyclone Gita,Cyclone,No,0,"Grey District Mayor Tony Kokshoorn, Christchur...",N/A,N/A,Impact of ex-Cyclone Gita on transportation an...
4,1/3/2022,OurNelson,Whakatū Nelson,Nelson flooding and Cyclone Gabrielle,Flooding and cyclone,Yes,5,Debs Martin (Chair of Climate Change Advisory ...,Dr. Anna Berthelsen (Marine Ecologist and Clim...,"""The Nelson flooding in August last year and c...",N/A
...,...,...,...,...,...,...,...,...,...,...,...
160,7/21/2021,NZHerald,West Coast,Severe weather,"Heavy rain, flooding",No,0,"Buller mayor Jamie Cleine, MetService forecast...",N/A,N/A,Severe weather causes damage and displacement ...
161,7/4/2017,NZHerald,Bay of Plenty,Flood,Flood,No,0,"Whakatane District mayor Tony Bonne, East Coas...",N/A,N/A,"Flooding in Bay of Plenty, recovery efforts un..."
162,3/2/2023,The Spinoff,Auckland,Torrential Rainfall,Extreme Rainfall,Yes,9,"PM Chris Hipkins, Mayor Wayne Brown, Sam Dean ...","Sam Dean, Friederike Otto","""Similarly, media reports increasingly note th...",The role of climate change in extreme weather ...
163,8/22/2018,Stuff,Nelson,Heavy Rain and Flooding,Heavy Rain and Flooding,No,0,"PM Jacinda Ardern, Emergency Management Minist...",N/A,N/A,"Heavy rain, flooding, road closures, evacuatio..."


In [117]:
#This saves the dataframe as a Pickle with the name of the corpus it includes

corpus_dataframe.to_pickle('{}.pkl'.format(corpus))

### Loading and Displaying the Dataframe
The next thing we want to do is to query the dataframe. To do this, we'll use something called an agent. 
The arguments we feed it are the llm, the corpus_data framework, and verbosity, which will give us it's chain of thought. We can turn this off for simpler answers. 

In [118]:
#Loading the corpus data from pickle. Don't run this if you've just build a new corpus!
loaded_corpus_dataframe = pd.read_pickle('Original Corpus.pkl')

In [119]:
corpus_df = loaded_corpus_dataframe # or corpus_data if you've built a new dataframe

corpus_df = corpus_df.sort_values('News Outlet')
display(HTML(corpus_df.to_html()))

,Date,News Outlet,Region Reported On,Name of Weather Event,Weather Type,Mention of 'climate change',Number of times 'climate change' is mentioned,Significant people (and roles) mentioned,Scientists mentioned,One sentence from text that includes 'climate change',Key theme (max 10 words)
74,3/15/2023,Diplomat,New Zealand,Cyclone Gabrielle,Cyclone,Yes,6,"Sam Dean (co-author and scientist at New Zealand's National Institute of Water and Atmospheric Research), Michael Mann (University of Pennsylvania climate scientist), Friederike Otto (climate scientist at Imperial College of London)",23 scientists from around the globe,"""Climate change is a serious concern for flooding in New Zealand and you’ve got to understand these are gigantic amounts of rainfall,"" said Sam Dean, a co-author and scientist at New Zealand's National Institute of Water and Atmospheric Research.",Climate change's role in worsening cyclone and flooding.
119,2/19/2018,Express,New Zealand,Cyclone Gita,Cyclone,No,0,N/A,N/A,N/A,Cyclone Gita causing travel disruptions in New Zealand.
10,7/21/2021,Guardian,"West Coast, New Zealand",Severe Flooding,"Heavy Rain, Storms",Yes,3,"Acting Minister for Emergency Management Kris Faafoi, Agriculture Minister Damien O'Connor, Prime Minister of the Netherlands Mark Rutte, Chancellor Angela Merkel",N/A,"""While no single flooding event can be directly attributed to the climate crisis, scientists have long warned that global heating would increase the number and likelihood of extreme weather events, including flooding, wildfires and heatwaves.""","Severe flooding and its impact, calls for action on climate change."
5,7/22/2017,Guardian,South Island,Severe Storm,Storm,No,0,"Civil Affairs Minister Nathan Guy, Prime Minister Bill English, Christchurch Mayor Lianne Dalziel",N/A,N/A,Severe storm causes evacuations and road closures in South Island.
153,4/21/2023,Mongabay,New Zealand,Cyclone Gabrielle,Cyclone,Yes,3,"Prime Minister Chris Hipkins, Renee Raroa (Ngati Porou Māori representative), Claire Charters (Māori Indigenous Rights Governance Partner at the New Zealand Human Rights Commission), Hannah McGlade (Indigenous Noongar member of the Permanent Forum from Australia)",N/A,"""With the frequency and severity of storms increasing, along with other climate impacts like rising sea levels, Māori peoples are facing increasingly dire climate crises and calling on the United Nations for help.""",Māori leaders call for inclusion in disaster recovery plans.
107,2/3/2023,NIWA,Auckland,January 2023 Rainfall,Rainfall,Yes,2,"Honorary Associate Professor Anthony Fowler, NIWA meteorologist Ben Noll, NIWA Climate Scientist Dr Sam Dean, NIWA Urban Aquatic Scientist Dr Annette Semadeni-Davies","Honorary Associate Professor Anthony Fowler, NIWA meteorologist Ben Noll, NIWA Climate Scientist Dr Sam Dean, NIWA Urban Aquatic Scientist Dr Annette Semadeni-Davies","""NIWA Climate Scientist Dr Sam Dean said this event was made more intense due to the influence of climate change.""","Record-breaking rainfall, impact of climate change on extreme events."
14,9/2/2022,NIWA,New Zealand,Extreme rainfall and flooding,"Heavy rain, atmospheric river",No,0,N/A,NIWA,N/A,"2nd-warmest August on record, extreme rainfall and flooding."
152,7/21/2021,NZ Herald,West Coast,Severe Flooding,Flood,No,0,"Acting Minister for Emergency Management Kris Faafoi, Local Government Minister Nanaia Mahuta, West Coast-Tasman MP Damien O'Connor",N/A,N/A,Government pledges $8 million for Westport flood recovery.
143,3/27/2017,NZ Herald,North Island,Flash flooding,Tropical deluge,No,0,Fire Service northern communications shift manager Colin Underdown,N/A,N/A,"Flash flooding in Auckland, more wet weather expected."
102,7/23/2020,NZ Herald,North Island,One-in-500 year rainfall event,Rainfall/Flooding,No,0,N/A,Weather experts,N/A,Severe rainfall event causing widespread damage and chaos.


# Querying the Dataframe
Using an 'agent' called pandas_dataframe_agent which converts natural language into commands that work on pandas dataframes.
This is interesting to play around with! 

In [122]:
question = "How many articles for each news outlet?"

agent = create_pandas_dataframe_agent(llm, corpus_df, verbose=True)
print(agent.run(question))



> Entering new AgentExecutor chain...
Thought: I need to group the dataframe by the 'News Outlet' column and then count the number of articles for each group.
Action: python_repl_ast
Action Input: df.groupby('News Outlet').count()
Observation:                      Date  Region Reported On  Name of Weather Event  \
News Outlet                                                            
Diplomat                1                   1                      1   
Express                 1                   1                      1   
Guardian                2                   2                      2   
Mongabay                1                   1                      1   
NIWA                    2                   2                      2   
NZ Herald              10                  10                     10   
NZHerald               28                  28                     28   
NewsHub                 1                   1                      1   
Newshub                 6         